# Entraînement du classifieur d'émotions (FER-2013), version 2 : entraînement long

Ce notebook annexe s'exécute dans **Google Colab avec un GPU**. Il apprend à reconnaître 7 émotions sur des photos de visages, puis exporte un fichier `predictions_test.csv` que le notebook principal du projet rechargera.

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*.

**Différences avec la version 1 :** jusqu'à 200 passages au lieu de 40 (l'entraînement s'arrête tout seul quand le score ne progresse plus pendant 15 passages), et les images sont aussi légèrement décalées pendant l'entraînement. Si le dossier `fer2013/` existe déjà dans cette session Colab, le zip n'est pas redemandé.

## 1. Importer le zip FER-2013

La cellule ci-dessous ouvre un bouton « Choisir les fichiers ». Sélectionne le zip FER-2013 téléchargé depuis Kaggle (environ 60 Mo). L'envoi peut prendre une vingtaine de minutes : laisse l'onglet ouvert.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

if Path("fer2013/train").exists():
    print("Le dossier fer2013/ existe déjà : pas besoin de renvoyer le zip.")
else:
    uploaded = files.upload()            # ouvre le bouton d'envoi
    zip_name = list(uploaded.keys())[0]  # nom du fichier envoyé
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("fer2013")
    print("Zip décompressé dans le dossier fer2013/")

## 2. Charger les images en mémoire

Chaque image est une photo de visage de 48×48 pixels en niveaux de gris, rangée dans un dossier qui porte le nom de son émotion. On parcourt les dossiers avec une boucle et on transforme chaque image en tableau de nombres.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path("fer2013")
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]


def load_split(split):
    """Charge toutes les images de 'train' ou 'test'.
    Retourne les pixels (X), le numéro d'émotion (y) et le chemin de chaque image."""
    images, labels, paths = [], [], []
    for label, emotion in enumerate(EMOTIONS):
        for path in sorted((DATA_DIR / split / emotion).glob("*.jpg")):
            images.append(np.array(Image.open(path).convert("L"), dtype="uint8"))
            labels.append(label)
            paths.append(f"{split}/{emotion}/{path.name}")
    return np.stack(images), np.array(labels), paths


X_train, y_train, paths_train = load_split("train")
X_test, y_test, paths_test = load_split("test")
print("Entraînement :", X_train.shape, "| Test :", X_test.shape)

In [ ]:
# Combien d'images par émotion ?
counts = pd.Series(y_train).map(lambda i: EMOTIONS[i]).value_counts()
print(counts)

# Un exemple de visage par émotion
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
for label, ax in enumerate(axes):
    ax.imshow(X_train[y_train == label][0], cmap="gray")
    ax.set_title(EMOTIONS[label])
    ax.axis("off")
plt.show()

## 3. Modèle de référence : régression logistique

Avant le réseau de neurones, on entraîne un modèle très simple qui regarde chaque pixel séparément. Il sert de point de comparaison : si le réseau de neurones ne fait pas nettement mieux, c'est qu'il n'apporte rien.

Pour que ce soit rapide, on l'entraîne sur 10 000 images tirées au hasard.

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Les pixels vont de 0 à 255 : on les ramène entre 0 et 1, et on aplatit chaque image en une ligne
X_train_flat = X_train.reshape(len(X_train), -1) / 255.0
X_test_flat = X_test.reshape(len(X_test), -1) / 255.0

rng = np.random.default_rng(42)
sample = rng.choice(len(X_train_flat), size=min(10_000, len(X_train_flat)), replace=False)

baseline = LogisticRegression(max_iter=100)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    baseline.fit(X_train_flat[sample], y_train[sample])

baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test_flat))
most_frequent = np.bincount(y_test).max() / len(y_test)
print(f"Toujours répondre 'happy'      : {most_frequent:.1%}")
print(f"Régression logistique (pixels) : {baseline_accuracy:.1%}")

## 4. Réseau de neurones convolutif (CNN)

Un CNN ne regarde pas les pixels un par un : il fait glisser de petits filtres sur l'image pour repérer des formes (un coin de bouche relevé, des sourcils froncés), puis combine ces formes pour décider de l'émotion.

- `Conv2D` : les filtres qui détectent les formes.
- `MaxPooling2D` : réduit la taille de l'image pour ne garder que l'essentiel.
- `Dropout` : éteint des neurones au hasard pendant l'entraînement, pour éviter le « par cœur ».
- `RandomFlip`, `RandomRotation`, `RandomZoom`, `RandomTranslation` : modifient légèrement les images à chaque passage, ce qui revient à avoir plus de données.
- La dernière couche `softmax` donne 7 probabilités, une par émotion, dont la somme vaut 1.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("GPU détecté :", bool(tf.config.list_physical_devices("GPU")))
keras.utils.set_random_seed(42)

# Format attendu par le réseau : (nombre d'images, 48, 48, 1), valeurs entre 0 et 1
X_train_cnn = (X_train / 255.0).astype("float32")[..., None]
X_test_cnn = (X_test / 255.0).astype("float32")[..., None]

# On met de côté 10 % des images d'entraînement pour surveiller l'apprentissage (validation)
order = np.random.default_rng(42).permutation(len(X_train_cnn))
n_val = len(order) // 10
val_idx, train_idx = order[:n_val], order[n_val:]


def conv_block(filters):
    return [
        layers.Conv2D(filters, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(filters, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
    ]


model = keras.Sequential(
    [
        layers.Input(shape=(48, 48, 1)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.08, 0.08),
        *conv_block(32),
        *conv_block(64),
        *conv_block(128),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(len(EMOTIONS), activation="softmax"),
    ]
)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# L'entraînement s'arrête tout seul quand le score de validation ne progresse plus
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]

history = model.fit(
    X_train_cnn[train_idx], y_train[train_idx],
    validation_data=(X_train_cnn[val_idx], y_train[val_idx]),
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
)

In [ ]:
# Courbes d'apprentissage : si la courbe 'validation' décroche de 'entraînement', le modèle apprend par cœur
hist = pd.DataFrame(history.history)
hist.to_csv("training_history.csv", index_label="epoch")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist["accuracy"], label="entraînement")
ax.plot(hist["val_accuracy"], label="validation")
ax.set_xlabel("Époque (passage complet sur les données)")
ax.set_ylabel("Taux de bonnes réponses")
ax.set_title("Apprentissage du CNN")
ax.legend()
fig.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Évaluation sur les images de test

Les images de test n'ont jamais été vues pendant l'entraînement : c'est la seule mesure honnête. La **matrice de confusion** montre, pour chaque émotion réelle (ligne), ce que le modèle a répondu (colonne).

In [ ]:
proba_test = model.predict(X_test_cnn, batch_size=256)   # 7 probabilités par image

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

pred_test = proba_test.argmax(axis=1)                    # l'émotion la plus probable
cnn_accuracy = accuracy_score(y_test, pred_test)
print(f"Régression logistique : {baseline_accuracy:.1%}")
print(f"CNN                   : {cnn_accuracy:.1%}")
print()
print(classification_report(y_test, pred_test, target_names=EMOTIONS, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test, display_labels=EMOTIONS, normalize="true",
    values_format=".0%", cmap="Blues", xticks_rotation=45, ax=ax, colorbar=False,
)
ax.set_xlabel("Émotion prédite")
ax.set_ylabel("Émotion réelle")
ax.set_title(f"Matrice de confusion du CNN (précision globale {cnn_accuracy:.1%})")
fig.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Export des résultats

On enregistre une ligne par image de test : son chemin, l'émotion réelle, l'émotion prédite et les 7 probabilités. C'est ce fichier que le notebook principal utilisera pour recommander des musiques.

In [ ]:
predictions = pd.DataFrame(proba_test, columns=[f"p_{e}" for e in EMOTIONS]).round(4)
predictions.insert(0, "image", paths_test)
predictions.insert(1, "true_emotion", [EMOTIONS[i] for i in y_test])
predictions.insert(2, "predicted_emotion", [EMOTIONS[i] for i in pred_test])
predictions.to_csv("predictions_test.csv", index=False)

pd.DataFrame(
    {"model": ["logistic_regression", "cnn"], "test_accuracy": [baseline_accuracy, cnn_accuracy]}
).to_csv("model_scores.csv", index=False)

predictions.head()

In [ ]:
model.save("emotion_model.keras")

# Télécharge les fichiers produits sur ton ordinateur (autorise les téléchargements multiples si le navigateur le demande)
for name in ["predictions_test.csv", "model_scores.csv", "training_history.csv",
             "confusion_matrix.png", "training_curves.png", "emotion_model.keras"]:
    files.download(name)